# Recitation 4 — Graph Algorithms (Java)

Save this notebook in your `cs301-notebooks` project and open it in your IDE.
Use the Java (IJava) kernel.
See [Recitation 0](https://harryc919.github.io/TA-CS301/recitation0.html) for setup.

Write your reasoning in the Markdown answer cells (double-click to edit, Shift+Enter to render).
For Exercise 3, edit and run the starter definition, then run its self-check.
An untouched starter fails the check; that is expected. Keep the signature unchanged.
The Floyd–Warshall section at the end is optional extra content and has its own starter and check.
Reference answers are not in this notebook; they are on the
[Recitation 4 page](https://harryc919.github.io/TA-CS301/recitation4.html). Nothing is submitted or graded.

IJava accepts class definitions and statements directly, without a main method. Each starter defines a class named `Solution`: re-run the matching starter cell before its check cell. A check failure throws an exception; no JVM assertion flag is needed.


## What we are practicing

This Tuesday session covers the graph-search lectures from the second half of Week 3 (BFS, DFS, UCS, IDDFS) and the Week 4 lectures on minimum spanning trees (Prim, Kruskal) and single-source shortest paths (Dijkstra, Bellman-Ford). Graph search is CS201 review, so it gets a one-table summary and no exercise. All-pairs shortest paths (Floyd–Warshall) appears only in the optional Extra section.

- Compare BFS, DFS, UCS, and IDDFS by where they keep waiting nodes, what they guarantee, and how much memory they use.
- State the cut property and explain Prim and Kruskal as two ways of applying it.
- Trace Prim and Kruskal, and decide, with a justification or a counterexample, whether an edge must, may, or cannot be in an MST.
- Explain which assumption Dijkstra relies on, and find the step where it fails on a graph with a negative edge.
- Explain what Bellman-Ford's rounds mean, why V − 1 rounds are enough, and how one more round detects a negative cycle.
- Implement Bellman-Ford with a limited number of rounds, and explain why each round must read the previous round's distances.

Prerequisites: adjacency lists and matrices, BFS and DFS from CS201, priority queues as used in lecture, and the exchange arguments from Recitation 3.

Spend about 12 minutes on the three review sections, then about 40 minutes on the core practice: Exercise 1 (12 minutes), Exercise 2 (10 minutes), and Exercise 3 (18 minutes). Leave about 10 minutes for discussion. Timing is a guide, not a completion requirement. The optional practice and the Floyd–Warshall section at the end are for after the session or for students who finish early.

These exercises are optional and ungraded. Use hints when needed. AI tools are allowed during recitation, but you should be able to explain, verify, and modify your answer. Follow the instructor's separate assignment and exam policies.

## 1. Graph search in one table

All four algorithms repeatedly take a waiting node, look at its neighbors, and add the new ones to the waiting set. They differ in *which* waiting node comes out next.

| Algorithm | Waiting nodes kept in | Guarantee | Memory |
| --- | --- | --- | --- |
| BFS | FIFO queue | Path with the fewest edges | O(V): a whole level can wait at once |
| DFS | Stack (or recursion) | None for path length | O(depth) along the current path |
| UCS | Priority queue keyed by path cost | Cheapest path, if all costs are non-negative | O(V) |
| IDDFS | Depth-limited DFS, repeated with limits 0, 1, 2, … | Path with the fewest edges, like BFS | O(depth), like DFS |

On an explicit graph, BFS and DFS take O(V + E) time. Two connections are worth remembering:

- **UCS is Dijkstra's algorithm that stops early**, as soon as the goal is removed from the priority queue. Section 3 explains why that removal is final.
- **IDDFS repeats the shallow levels** on every iteration, but when each node has at least two children the deepest level dominates the work, so the repetition costs only a constant factor. In exchange it keeps DFS's small memory.

## 2. Minimum spanning trees

Given a connected, undirected graph with edge weights, a *spanning tree* connects all V vertices with V − 1 edges and no cycle. A *minimum spanning tree* (MST) is one with the smallest total weight.

**Cut property.** Split the vertices into two sides S and V − S. A lightest edge crossing the split belongs to some MST. When all weights are distinct, it belongs to every MST, and the MST is unique.

Proof idea, an exchange argument as in Recitation 3: take an MST T that does not contain the lightest crossing edge e. Adding e to T creates a cycle, and that cycle must cross the split a second time through some edge f. Replacing f by e gives another spanning tree, and since w(e) ≤ w(f) its weight is no larger. So some MST contains e.

- **Prim** grows one tree from a start vertex. The split is (tree, everything else), and each step adds the lightest edge leaving the tree. With a binary-heap priority queue it runs in O(E log V).
- **Kruskal** scans all edges from lightest to heaviest and adds an edge whenever it joins two different components, so it never closes a cycle. The split for an accepted edge is one of its two components against everything else. Sorting costs O(E log E) = O(E log V). Checking whether two vertices are already in the same component needs a separate data structure (union-find, not covered in this recitation); with it, the checks add almost nothing and the total is O(E log V).

Neither algorithm cares whether weights are negative: the cut property only compares weights of edges that cross the same split.

## 3. Single-source shortest paths

Given a weighted directed graph and a source s, find the distance d(v) of the cheapest path from s to every vertex v. Both algorithms in this section keep a tentative distance `d[v]` (∞ at the start, 0 for s) and improve it with the same step, called *relaxing* the edge (u, v):

```pseudocode
RELAX(u, v, w)
    if d[u] + w < d[v]:
        d[v] = d[u] + w     // the path to u, then edge (u, v), is cheaper
```

### Dijkstra: finalize the closest vertex

```pseudocode
DIJKSTRA(G, s)
    d[v] = ∞ for every vertex v;  d[s] = 0
    Q = priority queue of all vertices, keyed by d
    while Q is not empty:
        u = EXTRACT-MIN(Q)          // d[u] is now final
        for each edge (u, v, w):
            if v is still in Q:     // never update a final vertex
                RELAX(u, v, w)      // and decrease v's key in Q
```

Why removing u makes d[u] final: when u comes out, every other path to u has to leave the set of final vertices at some vertex x that is still in Q, and d[x] ≥ d[u] because u was the minimum. If all weights are **non-negative**, the rest of that path cannot make it cheaper than d[u]. With a binary heap, Dijkstra runs in O((V + E) log V).

### Bellman-Ford: relax everything, V − 1 times

```pseudocode
BELLMAN-FORD(G, s)
    d[v] = ∞ for every vertex v;  d[s] = 0
    repeat V − 1 times:             // one "round"
        for each edge (u, v, w):
            RELAX(u, v, w)
    for each edge (u, v, w):        // one extra check
        if d[u] + w < d[v]:
            return "negative cycle reachable from s"
    return d
```

**Round invariant:** after round i, `d[v]` is at most the cost of the cheapest path from s to v that uses at most i edges. Without a negative cycle, some shortest path repeats no vertex, so it has at most V − 1 edges, and V − 1 rounds are enough. If the extra check can still improve an edge, a negative cycle is reachable from s. Bellman-Ford runs in O(VE) time and handles negative edges.

## Core practice · about 40 minutes

Try each prompt before opening the reference explanation. Written answers can be short lists, small tables, or sentences. Use your IDE or the downloadable notebook to run code: the editors on this page display and copy code but do not run or grade it.

1. Exercise 1 · MST traces and true/false — about 12 minutes.
2. Exercise 2 · Dijkstra vs Bellman-Ford — about 10 minutes.
3. Exercise 3 · cheapest route with at most k stops — about 18 minutes.

> Browser code drafts are temporary. Copy your work before leaving the page. Tab indents, Shift+Tab unindents, and Escape then Tab leaves the editor. Refresh restores templates; Reset code restores the selected language's template.

## Exercise 1 · MST traces and true/false

**Two traces and three claims · about 12 minutes.** Use this undirected graph with distinct weights. Its edges are A–B 4, A–C 1, B–C 3, B–D 2, C–D 5, C–E 8, D–E 6, and D–F 9.

1. **Kruskal.** List the edges in the order Kruskal accepts them. Name the first edge it rejects and the cycle that edge would close. Give the total weight.
2. **Prim from A.** At each step, write the edge Prim chooses. Is the tree the same as Kruskal's? Is the order the same? Explain why the trees must agree for this graph.
3. **True or false?** Justify each claim below with a short argument or a counterexample.

- **3a.** The heaviest edge of a connected graph is never in an MST.
- **3b.** If all weights are distinct, the lightest edge is in every MST.
- **3c.** An MST contains a shortest path between every pair of vertices.

<details><summary>Hint 1 · Use the graph itself for 3a</summary>

Look at vertex F. Which edges reach it, and could a spanning tree leave both of them out?

</details>

<details><summary>Hint 2 · Three vertices are enough for 3c</summary>

Build a triangle where the direct edge between two vertices is a bit cheaper than going around, but the other two edges are cheaper still.

</details>

### Your answer — Exercise 1

1. **Kruskal:** Accepted edges in order, the first rejected edge and its cycle, and the total weight.
2. **Prim from A:** The edge chosen at each step. Same tree? Same order? Why must the trees agree?
3. **True or false:**
   - 3a. Heaviest edge never in an MST:
   - 3b. Lightest edge in every MST (distinct weights):
   - 3c. MST contains shortest paths:


## Exercise 2 · Dijkstra vs Bellman-Ford

**A trace and two explanations · about 10 minutes.** Use this directed graph with edges S→A 2, S→B 5, B→A −4, and A→T 1. The source is S.

1. **Trace Dijkstra** exactly as in the pseudocode of Section 3: a vertex is final when it is removed, and a final vertex is never updated. Give the order in which vertices become final and the final value of d for every vertex. Then find the true distances by hand.
2. **Diagnose.** At which step did Dijkstra go wrong, and which assumption from the correctness argument failed?
3. **Bellman-Ford.** Run it on the same graph with edges in the order S→A, S→B, B→A, A→T. Then explain in general why V − 1 rounds are enough when there is no negative cycle, and how the extra check detects a negative cycle.

<details><summary>Hint 1 · Write d after every removal</summary>

Keep a row of tentative distances for S, A, B, T, and update it each time a vertex comes out of the queue. Mark the vertices that are final. When B comes out, look at the edge B→A.

</details>

<details><summary>Hint 2 · How many edges can a shortest path have?</summary>

If a shortest path visited some vertex twice, the part between the two visits is a cycle. What is the cost of that cycle if there is no negative cycle, and what happens when you cut it out?

</details>

### Your answer — Exercise 2

1. **Dijkstra trace:** Order of finalized vertices, final d values, and the true distances.

| Removed (final) | d[S] | d[A] | d[B] | d[T] |
|---|---|---|---|---|
| Start | 0 | ∞ | ∞ | ∞ |
| | | | | |
| | | | | |
| | | | | |
| | | | | |

2. **Diagnosis:** The wrong step and the failed assumption.
3. **Bellman-Ford:** Your run on this graph; why V − 1 rounds suffice; how the extra check detects a negative cycle.


## Exercise 3 · Cheapest route with at most k stops

**Coding and explanation · about 18 minutes.** There are n cities, numbered 0 to n − 1, and a list of one-way flights `[from, to, price]`. Return the cheapest total price of a route from `src` to `dst` that makes **at most k stops**, that is, uses at most k + 1 flights. Return −1 if no such route exists.

This is Bellman-Ford with k + 1 rounds instead of V − 1: by the round invariant from Section 3, after round t the distances account for every route with at most t flights. There is one catch. The textbook Bellman-Ford updates `d` in place, so a value improved early in a round can be used again later in the *same* round, and one round can chain several flights. That speeds up the unrestricted version, but here it would let a route use more flights than allowed. Each round must therefore read from a copy of the distances as they were at the start of the round.

- 1 ≤ n ≤ 100; 0 to 10,000 flights; `from ≠ to`; each price is between 0 and 10,000; several flights may join the same pair of cities.
- `src` and `dst` are valid cities, and 0 ≤ k ≤ n − 1. If `src == dst`, the answer is 0.
- Every answer is at most 1,000,000, so a 32-bit integer is enough. In C, C++, C#, and Java the starter uses the largest `int` as ∞, so never add a price to an unreached city's ∞.
- Do not modify `flights`. Target: O(k · m) time for m flights and O(n) extra space.

### Examples and checks

| Flights | Route | k | Answer | What to check |
| --- | --- | --- | --- | --- |
| 0→1 100, 1→2 100, 2→0 100, 1→3 600, 2→3 200 | 0 to 3 | 1 | 700 | 0→1→3; the cheaper 0→1→2→3 needs two stops |
| Same flights | 0 to 3 | 2 | 400 | Now 0→1→2→3 is allowed |
| 0→1 100, 1→2 100, 0→2 500 | 0 to 2 | 0 | 500 | An in-place round returns 200 here |
| Same flights | 0 to 2 | 1 | 200 | One stop at city 1 |
| 0→1 1, 1→2 1, 2→3 1 | 0 to 3 | 1 | −1 | A route exists, but needs two stops |
| Any | 0 to 0 | 0 | 0 | Source equals destination |

### Function interface (Java)

```java
public static int cheapestRoute(int n, int[][] flights, int src, int dst, int k)
```

**Explain your answer:** why does reading from a copy guarantee that round t only adds routes with exactly one more flight? Why are k + 1 rounds the right number? Give the time and space. Finally, why would plain Dijkstra on price, ignoring the number of flights, fail on the trap example with k = 0?

<details><summary>Hint 1 · One round, step by step</summary>

At the start of a round, copy `dist` into `newDist`. For every flight (u, v, price), read `dist[u]` (the old value) and, if it is finite, try to lower `newDist[v]` to `dist[u] + price`. At the end of the round, replace `dist` with `newDist`.

</details>

<details><summary>Hint 2 · Test the trap by hand</summary>

With flights 0→1 100, 1→2 100, 0→2 500 and k = 0, there is exactly one round. If you write into the same array you read from, relaxing 0→1 sets city 1 to 100, and then 1→2 immediately reads that new value and sets city 2 to 200: two flights in one round. Reading the old copy keeps city 1 at ∞ for the whole round.

</details>

Edit and run the starter definition in the next cell, then run its self-check.

In [ ]:
public class Solution {
    public static int cheapestRoute(int n, int[][] flights, int src, int dst, int k) {
        final int INF = Integer.MAX_VALUE;
        // dist[v] = cheapest price to reach v using the flights allowed so far.
        int[] dist = new int[n];
        java.util.Arrays.fill(dist, INF);
        dist[src] = 0;
        // At most k stops means at most k + 1 flights: one round per flight.
        for (int round = 0; round <= k; round++) {
            // TODO: relax every flight {u, v, price} for this round.
            // Read this round's starting distances, and do not let one round
            // chain two flights.
        }
        // TODO: return the price of reaching dst, or -1 if it is unreachable.
        throw new UnsupportedOperationException();
    }
}


In [ ]:
// Re-run the Solution definition cell before running this check cell.
// These checks throw while the starter is still unchanged.
int[][] sample = {{0, 1, 100}, {1, 2, 100}, {2, 0, 100}, {1, 3, 600}, {2, 3, 200}};
int[][] trap = {{0, 1, 100}, {1, 2, 100}, {0, 2, 500}};
int[][] chain = {{0, 1, 1}, {1, 2, 1}, {2, 3, 1}};
Object[][] cases = {
    {4, sample, 0, 3, 1, 700, "sample, k = 1"},
    {4, sample, 0, 3, 2, 400, "sample, k = 2"},
    {4, sample, 0, 0, 0, 0, "src == dst"},
    {3, trap, 0, 2, 0, 500, "in-place trap: one round must not chain two flights"},
    {3, trap, 0, 2, 1, 200, "trap, k = 1"},
    {3, new int[][]{{0, 1, 5}}, 0, 2, 2, -1, "unreachable destination"},
    {4, chain, 0, 3, 1, -1, "route needs too many stops"},
    {4, chain, 0, 3, 2, 3, "chain, k = 2"},
    {2, new int[][]{{0, 1, 7}, {0, 1, 3}}, 0, 1, 0, 3, "parallel flights"},
    {3, new int[][]{{0, 1, 1}, {1, 0, 1}, {1, 2, 10}}, 0, 2, 2, 11, "cycle"},
    {3, new int[][]{{0, 1, 0}, {1, 2, 0}}, 0, 2, 1, 0, "zero prices"},
};
for (Object[] c : cases) {
    int[][] flights = (int[][]) c[1];
    String before = java.util.Arrays.deepToString(flights);
    int result = Solution.cheapestRoute((int) c[0], flights, (int) c[2], (int) c[3], (int) c[4]);
    if (result != (int) c[5])
        throw new AssertionError(c[6] + ": expected " + c[5] + ", got " + result);
    if (!java.util.Arrays.deepToString(flights).equals(before))
        throw new AssertionError(c[6] + ": flights were modified");
}
int[][] large = new int[100 * 99][];
int index = 0;
for (int u = 0; u < 100; u++)
    for (int v = 0; v < 100; v++)
        if (u != v) large[index++] = new int[]{u, v, (u * 37 + v * 91) % 10001};
if (Solution.cheapestRoute(100, large, 0, 99, 99) != 2464)
    throw new AssertionError("Check the 100-city, 9,900-flight case");
System.out.println("Exercise 3 checks passed (11 cases + 100 cities, 9,900 flights, k = 99).");


### Your explanation — Exercise 3

- **The copy:** Why does reading from a copy mean round t adds exactly one more flight?
- **Rounds:** Why are k + 1 rounds the right number?
- **Cost:** Time and extra space.
- **Plain Dijkstra:** Why does it fail on the trap example with k = 0?


## Optional practice

Two short written questions for after the session, or for students who finish early. They are not on the core path.

1. **Shift every weight?** A classmate proposes: "To handle negative edges, add the same constant to every weight so that all weights become non-negative, then run Dijkstra." Give a small counterexample, and explain what goes wrong in general.
2. **MST paths and bottlenecks.** In Exercise 1 (3c), MST paths are not always shortest. Show instead that the path between two vertices in an MST (distinct weights) makes its *heaviest* edge as light as possible: no other path between the same two vertices has a lighter heaviest edge.

<details><summary>Hint · Count the edges on each path</summary>

For question 1, adding c to every weight adds c × (number of edges) to every path. Compare a one-edge path with a three-edge path. For question 2, suppose another path had every edge lighter than the MST path's heaviest edge e, remove e from the MST, and look at the split that creates.

</details>

### Your answer — Optional practice (optional)

1. **Shifted weights:** Your counterexample, and what goes wrong in general.
2. **Bottleneck paths:** Your argument.


## Discussion and exit check

Compare your counterexamples from Exercise 1 and from the optional practice. Which ones needed a specific graph, and which ones would work with almost any graph of the right shape?

Prim and Dijkstra look almost identical: both grow a set of vertices and repeatedly take the best vertex from a priority queue. What exactly does each one put in the key, and why does only one of them care about negative weights?

In Exercise 3, writing in place was a bug. In the standard Bellman-Ford it is harmless. What is the difference between the two problems?

**Exit check:** a friend runs Prim on a graph with some negative edge weights, and Dijkstra on the same graph from some source. Which result can you trust, and why?

Before leaving, keep your own copy of the notebook or code, and write down one remaining question for the TA.

### Your reflection

**Prim vs Dijkstra:** What is each key, and why does only Dijkstra care about negative weights?

**In-place rounds:** Why is writing in place a bug in Exercise 3 but harmless in standard Bellman-Ford?

**Exit check:** Which result can you trust, Prim's or Dijkstra's, on a graph with negative edges, and why?


## Extra · All-pairs shortest paths with Floyd–Warshall

This section is provided by the TA for students who want to go further. It is not covered in lectures or on the final exam, and the core exercises do not depend on it. It solves all-pairs shortest paths as a dynamic program, using the same five steps as Recitation 3.

### Why we need it

Sometimes we want the distance between *every* pair of vertices, for example to fill a distance table between cities. Running Dijkstra from every vertex is fast but fails with negative edges. Running Bellman-Ford from every vertex handles negative edges but costs O($V^{2}$E), which is O($V^{4}$) on a dense graph. Floyd–Warshall handles negative edges (but not negative cycles) in Θ($V^{3}$) time with three short loops.

### The DP, step by step

Number the vertices 0 to n − 1.

1. **Subproblem.** $D_{k}$(i, j) is the cost of the cheapest path from i to j whose *intermediate* vertices all come from {0, …, k − 1}. The endpoints i and j themselves may be anything.
2. **Key math function.** The cheapest such path either avoids vertex k − 1, or passes through it exactly once (with no negative cycle, a shortest path never needs to repeat it). So $D_{k}$(i, j) = min( $D_{k-1}$(i, j), $D_{k-1}$(i, k − 1) + $D_{k-1}$(k − 1, j) ).
3. **Base case and order.** $D_{0}$(i, j) allows no intermediate vertex: it is the weight of the lightest edge i → j, 0 when i = j, and ∞ if there is no edge. Fill rounds k = 1, 2, …, n in order.
4. **Answer.** $D_{n}$(i, j) allows every vertex, so it is the true distance. One n × n table updated in place is enough: during round k, the row and column of vertex k − 1 do not change, because going through k − 1 to reach k − 1 itself adds D(k − 1, k − 1) = 0.
5. **Cost.** n rounds × $n^{2}$ cells × O(1) work = Θ($n^{3}$) time, and Θ($n^{2}$) space. A negative value on the diagonal at the end means a negative cycle.

```pseudocode
FLOYD-WARSHALL(n, edges)
    D[i][j] = 0 if i = j, else ∞
    for each edge (u, v, w):
        D[u][v] = min(D[u][v], w)        // lightest parallel edge
    for k = 0 to n − 1:                  // now allow vertex k in the middle
        for i = 0 to n − 1:
            for j = 0 to n − 1:
                if D[i][k] + D[k][j] < D[i][j]:
                    D[i][j] = D[i][k] + D[k][j]
    return D
```

In the code, the loop variable k is the vertex being allowed, so iteration k turns $D_{k}$ into $D_{k+1}$. Because ∞ is a large number in code, skip the update whenever D[i][k] or D[k][j] is ∞. Otherwise ∞ plus a negative weight could look like a real, finite distance.

### Worked example

Use this directed graph with edges 0→1 3, 0→3 7, 1→0 8, 1→2 2, 2→0 5, 2→3 1, and 3→0 2.

| Round | Allows in the middle | Entries that improve |
| --- | --- | --- |
| Start | Nothing (direct edges) | Rows: 0: [0, 3, ∞, 7]; 1: [8, 0, 2, ∞]; 2: [5, ∞, 0, 1]; 3: [2, ∞, ∞, 0] |
| k = 0 | Vertex 0 | D[1][3] = 15 (1→0→3), D[2][1] = 8 (2→0→1), D[3][1] = 5 (3→0→1) |
| k = 1 | Vertices 0, 1 | D[0][2] = 5 (0→1→2), D[3][2] = 7 (3→0→1→2) |
| k = 2 | Vertices 0–2 | D[0][3] = 6 (0→1→2→3), D[1][0] = 7 (1→2→0), D[1][3] = 3 (1→2→3) |
| k = 3 | All vertices | D[1][0] = 5 (1→2→3→0), D[2][0] = 3 (2→3→0), D[2][1] = 6 (2→3→0→1) |

| Final D | to 0 | to 1 | to 2 | to 3 |
| --- | --- | --- | --- | --- |
| from 0 | 0 | 3 | 5 | 6 |
| from 1 | 5 | 0 | 2 | 3 |
| from 2 | 3 | 6 | 0 | 1 |
| from 3 | 2 | 5 | 7 | 0 |

Notice that D[1][0] improves twice: first to 7 when vertex 2 is allowed, then to 5 when vertex 3 is allowed as well. Each round can only use paths through the vertices allowed so far.

## Extra · Build Floyd–Warshall

**Optional coding.** Given n vertices and a list of directed edges `[from, to, weight]`, return the n × n matrix `D` with `D[i][j]` = the shortest-path distance from i to j.

- 1 ≤ n ≤ 200; 0 to 10,000 edges; `from ≠ to`; each weight is between −10,000 and 10,000; parallel edges are allowed, and the lightest one counts. There is no negative cycle.
- `D[i][i] = 0`. If j cannot be reached from i, `D[i][j] = INF`, where every starter defines `INF = 1,000,000,000`. Never add to an `INF` entry.
- Do not modify `edges`. Target: Θ($n^{3}$) time and Θ($n^{2}$) space.

### Function interface (Java)

```java
public static int[][] floydWarshall(int n, int[][] edges)
```

**Going further (optional):** store a `next[i][j]` table so you can print the vertices of each shortest path, not just its cost. How would you report every pair whose distance is −∞ because a negative cycle lies on some path between them?

<details><summary>Hint 1 · Loop order matters</summary>

The loop over k must be the *outermost* loop: round k needs every value from round k − 1 for all pairs. With k innermost, D[i][j] would be finished before the middle vertices it depends on have been processed.

</details>

<details><summary>Hint 2 · Test the INF guard</summary>

Take n = 3 with a single edge 1→2 of weight −5. Vertex 0 reaches nothing, so D[0][2] must stay INF. Without the guard, round k = 1 computes D[0][1] + D[1][2] = INF − 5, which is less than INF.

</details>

Edit and run the starter definition in the next cell, then run its self-check.

In [ ]:
public class Solution {
    public static final int INF = 1_000_000_000;

    public static int[][] floydWarshall(int n, int[][] edges) {
        // Base case D_0: 0 on the diagonal, INF elsewhere.
        int[][] D = new int[n][n];
        for (int i = 0; i < n; i++) {
            for (int j = 0; j < n; j++) D[i][j] = i == j ? 0 : INF;
        }
        // TODO: load each edge {u, v, w}, keeping the lightest of parallel edges.
        // TODO: for k, then i, then j, apply the key math function in place.
        // Never add to an INF entry.
        throw new UnsupportedOperationException();
    }
}


In [ ]:
// Re-run the Solution definition cell before running this check cell.
// These checks throw while the starter is still unchanged.
final int I = 1_000_000_000;  // INF
Object[][] cases = {
    {4, new int[][]{{0, 1, 3}, {0, 3, 7}, {1, 0, 8}, {1, 2, 2}, {2, 0, 5}, {2, 3, 1}, {3, 0, 2}},
     new int[][]{{0, 3, 5, 6}, {5, 0, 2, 3}, {3, 6, 0, 1}, {2, 5, 7, 0}}, "worked example"},
    {1, new int[0][], new int[][]{{0}}, "one vertex"},
    {3, new int[][]{{0, 1, 4}, {0, 2, 5}, {2, 1, -3}}, new int[][]{{0, 2, 5}, {I, 0, I}, {I, -3, 0}}, "negative edge"},
    {3, new int[][]{{0, 1, 1}}, new int[][]{{0, 1, I}, {I, 0, I}, {I, I, 0}}, "unreachable pairs"},
    {3, new int[][]{{1, 2, -5}}, new int[][]{{0, I, I}, {I, 0, -5}, {I, I, 0}}, "never add to INF"},
    {2, new int[][]{{0, 1, 5}, {0, 1, 2}, {1, 0, 4}}, new int[][]{{0, 2}, {4, 0}}, "parallel edges"},
};
for (Object[] c : cases) {
    int[][] edges = (int[][]) c[1];
    String before = java.util.Arrays.deepToString(edges);
    int[][] result = Solution.floydWarshall((int) c[0], edges);
    if (!java.util.Arrays.deepEquals(result, (int[][]) c[2]))
        throw new AssertionError(c[3] + ": got " + java.util.Arrays.deepToString(result));
    if (!java.util.Arrays.deepToString(edges).equals(before))
        throw new AssertionError(c[3] + ": edges were modified");
}
int n = 200;
java.util.List<int[]> list = new java.util.ArrayList<>();
for (int i = 0; i < n; i++) list.add(new int[]{i, (i + 1) % n, 1});
for (int i = 0; i < n; i++) if ((i * 7 + 3) % n != i) list.add(new int[]{i, (i * 7 + 3) % n, n + i % 5});
int[][] large = Solution.floydWarshall(n, list.toArray(new int[0][]));
for (int i = 0; i < n; i++)
    for (int j = 0; j < n; j++)
        if (large[i][j] != ((j - i) % n + n) % n) throw new AssertionError("Check the 200-vertex case");
System.out.println("Floyd-Warshall checks passed (6 cases + 200 vertices).");


## References and related material

- T. H. Cormen, C. E. Leiserson, R. L. Rivest, and C. Stein, *Introduction to Algorithms*, 4th ed. (MIT Press, 2022): Chapter 20 (Elementary Graph Algorithms), Chapter 21 (Minimum Spanning Trees), Chapter 22 (Single-Source Shortest Paths), and Chapter 23 (All-Pairs Shortest Paths).
- S. Russell and P. Norvig, *Artificial Intelligence: A Modern Approach*, 4th ed. (Pearson, 2020): Chapter 3, for uniform-cost search and iterative deepening.
- [Recitation 3 · greedy proofs and the cut property](https://harryc919.github.io/TA-CS301/recitation3.html#greedy-review) — the exchange argument behind the MST proofs.
- [Recitation 3 · a common path for DP problems](https://harryc919.github.io/TA-CS301/recitation3.html#dp-review) — the five steps used for Floyd–Warshall.
- [Recitation 0 · notebook environment and kernels](https://harryc919.github.io/TA-CS301/recitation0.html)